# f(y_hat) vs f(y_tilde) loss placement

Trains PAL on a 2D toy problem with the objective evaluated at y_hat, at the projected y_tilde, or switched midway, and compares residuals, objective and gradient norms.

In [ ]:
BATCH_SIZE = 100
DEPTH = 3
WIDTH = 128
EPOCHS = 5000

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)


def example_problem(x, y):
    assert x.shape == (BATCH_SIZE, 2), f"x shape: {x.shape} instead of (BATCH_SIZE, 2)"
    assert y.shape == (BATCH_SIZE, 2), f"y shape: {y.shape} instead of (BATCH_SIZE, 2)"

    # objective: per-sample squared distance, averaged over batch
    f = ((y[:, 0]-0.5) ** 2 + (y[:, 1] - 0.5) ** 2).mean()                                      

    # equality constraints: h_i(y) = 0
    h1_raw = 10 * y[:, 0]                            # linear - 1-step Newton is ~ exact 
    h2_raw = 0.01 * torch.sin(y[:, 1] * 5) + 0.1 * y[:, 0]     #  non-linear
    h1 = h1_raw.mean()
    h2 = h2_raw.mean()
    
    return f, h1, h2, h1_raw, h2_raw


def model():
    layers = []
    for i in range(DEPTH):
        in_dim = 2 if i == 0 else WIDTH
        layers += [nn.Linear(in_dim, WIDTH), nn.ReLU()]
    layers += [nn.Linear(WIDTH, 2)]
    return nn.Sequential(*layers)


def sample_conditions(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.randn(BATCH_SIZE, 2, generator=generator)

In [ ]:
t = torch.linspace(-1, 1, BATCH_SIZE)                                                                                 
                                
x = torch.stack([t, t], dim=1)                                                                                        
y = torch.stack([t, -t], dim=1)                        
                                                                                                                    
f, h1, h2, h1_raw, h2_raw = example_problem(x, y)      

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))                                                                        

ax[0].plot(t, h1_raw.detach())                                                                                        
ax[0].axhline(0, c='k', lw=.8, ls='--')                
ax[0].set(title="h1 = 10*y[:, 0]",            xlabel="y0", ylabel="h1")                                               
ax[0].grid(alpha=.3)                                                                                                  
                                                                                                                    
ax[1].plot(t, h2_raw.detach(), c='C1')                                                                                
ax[1].axhline(0, c='k', lw=.8, ls='--')                
ax[1].set(title="h2 = 0.01*sin(5*y[:, 1]) + 0.1*y[:, 0]",   xlabel="y1", ylabel="h2")                                               
ax[1].grid(alpha=.3)                                                                                                  

plt.tight_layout()                                                                                                    
plt.show()   

In [ ]:
def lm_project_step(y, constraint_stack_fn, *,                                                                        
                    delta=1e-3, lam_min=1e-6,                                                                         
                    prescale=True, prescale_floor=1.0):                                                               
    """                                                                                                               
    One K-space Levenberg-Marquardt step with Yamashita-Fukushima damping.                                            
    Equivalent to pal/projection/projector.py:_project_step_inner_lm_k for                                            
    the equality-only case (all constraints active).                                                                  
                                                                                                                
    y:                   [B, D]  live iterate (grad flows to theta).                                                  
    constraint_stack_fn: callable(y) -> [B, K]  stacked residuals.                                                    
                                                                                                                    
    Returns:                                                                                                          
    y_new:             [B, D]  detached  (one LM step from y).                                                      
    c_pre:             [B, K]  live      (= constraint_stack_fn(y)).                                                
    c_post_live:       [B, K]  live      (residuals at y_tilde_live).                                               
    disp_live:         [B, D]  live      (y - y_tilde_live).                                                        
    """                                                                                                               
    B, D = y.shape                                                                                                    
                                                                                                                    
    # Jacobian via autograd on detached y (no 2nd-order terms reach theta).
    y_det = y.detach().requires_grad_(True)                                                                           
    c_det = constraint_stack_fn(y_det)                                                                                
    K = c_det.shape[-1]                                
    J = torch.zeros(B, K, D)                                                                                          
    for k in range(K):                                                                                                
        J[:, k, :] = torch.autograd.grad(
            c_det[:, k].sum(), y_det, retain_graph=(k < K - 1),                                                       
        )[0]                                                                                                          
    J = J.detach()
                                                                                                                    
    # Live c at y (for loss + displacement path).
    c_pre = constraint_stack_fn(y)          # live                                                                    
    c_pre_det = c_pre.detach()                                                                                        
                                                                                                                    
    # Row prescale (unit-norm rows, consistent c scaling).
    B_mat, c_masked = J, c_pre_det                                                                                    
    if prescale:                                                                                                      
        row_norms = B_mat.norm(dim=2, keepdim=True).clamp(min=prescale_floor)
        B_mat    = B_mat / row_norms                                                                                  
        c_masked = c_masked / row_norms.squeeze(-1)    
                                                                                                                    
    # YF damping: lam = max(delta * ||c||^2, lam_min), per sample.
    BBT       = torch.bmm(B_mat, B_mat.transpose(1, 2))
    c_norm_sq = c_masked.pow(2).sum(dim=1, keepdim=True)                                                              
    lam_lm    = (delta * c_norm_sq).clamp(min=lam_min)                                                                
    eye       = torch.eye(K, dtype=BBT.dtype, device=BBT.device)                                                      
    BBT_reg   = BBT + lam_lm.unsqueeze(-1) * eye                                                                      
                                                                                                                    
    # Detached step (for y_new).
    L  = torch.linalg.cholesky(BBT_reg)                                                                               
    z  = torch.cholesky_solve(c_masked.unsqueeze(-1), L).squeeze(-1)                                                  
    dy = torch.bmm(B_mat.transpose(1, 2), z.unsqueeze(-1)).squeeze(-1)                                                
    y_new = y.detach() - dy                                                                                           
                                                                                                                    
    # Live displacement: same L & B_mat (detached), but live c gives
    #     gradient through y -> c -> z_live -> disp_live -> y_tilde_live.                                             
    c_live = c_pre / row_norms.squeeze(-1).clamp(min=prescale_floor) if prescale else c_pre                           
    z_live      = torch.cholesky_solve(c_live.unsqueeze(-1), L).squeeze(-1)                                           
    disp_live   = torch.bmm(B_mat.transpose(1, 2), z_live.unsqueeze(-1)).squeeze(-1)                                  
    y_tilde_live = y - disp_live                                                                                      
    c_post_live  = constraint_stack_fn(y_tilde_live)   
                                                                                                                    
    return y_new, c_pre, c_post_live, disp_live 

PAL trainer with a switch for where the objective is evaluated, plus gradient diagnostics on a fixed eval batch.

In [ ]:
# loss_at = "yhat":   loss = f(y_hat)         + lam_c * |c(y_hat)| + lam_d * ||disp||  (PAL default)
# loss_at = "ytilde": loss = f(y_tilde_live)  + lam_c * |c(y_hat)| + lam_d * ||disp||  (broken placement)
# loss_at = "switch": start as "yhat", switch to "ytilde" at epoch `switch_at`.
#
# The constraint and displacement terms are identical between variants. Only
# the objective term moves. Tracks gradient diagnostics on a fixed eval batch:
#   * cos(grad_theta f(y_hat), grad_theta f(y_tilde))         -- direction overlap
#   * ||P_N grad_y f|| / ||grad_y f||                          -- fraction f(y_tilde) loses

def gated_ema_update3(lam, metric_mean, tol):
    PAL_EMA_RATE = 1e-2
    PAL_EMA_MAX_DECADES = 1
    gap = torch.log10(metric_mean.clamp(min=1e-30) / tol).clamp(max=PAL_EMA_MAX_DECADES)
    return (lam + PAL_EMA_RATE * gap).clamp(min=0.0)


def grad_diagnostics(net, x_e, cfn_e, proj_delta, proj_lam_min):
    """Return (cos_sim, normal_frac, |g_yhat|, |g_ytilde|) at current theta."""
    for p in net.parameters():
        if p.grad is not None: p.grad.zero_()
    y_hat = net(x_e)
    f_yhat, *_ = example_problem(x_e, y_hat)
    f_yhat.backward()
    g_yhat = torch.cat([p.grad.detach().clone().flatten() for p in net.parameters()])

    for p in net.parameters():
        if p.grad is not None: p.grad.zero_()
    y_hat = net(x_e)
    _, _, _, disp = lm_project_step(y_hat, cfn_e, delta=proj_delta, lam_min=proj_lam_min)
    y_tilde = y_hat - disp
    f_ytil, *_ = example_problem(x_e, y_tilde)
    f_ytil.backward()
    g_ytil = torch.cat([p.grad.detach().clone().flatten() for p in net.parameters()])

    cos = (g_yhat @ g_ytil) / (g_yhat.norm() * g_ytil.norm() + 1e-12)

    # ||P_N grad_y f|| / ||grad_y f|| at y_hat
    with torch.no_grad():
        y_hat_val = net(x_e).clone()
    y_d = y_hat_val.detach().requires_grad_(True)
    f_per = (y_d[:, 0] - 0.5).pow(2) + (y_d[:, 1] - 0.5).pow(2)
    grad_y = torch.autograd.grad(f_per.sum(), y_d)[0]

    y_J = y_hat_val.detach().requires_grad_(True)
    c_J = cfn_e(y_J)
    Bsz, K = c_J.shape
    D = y_J.shape[-1]
    J = torch.zeros(Bsz, K, D)
    for k in range(K):
        J[:, k, :] = torch.autograd.grad(c_J[:, k].sum(), y_J, retain_graph=(k < K - 1))[0]
    J = J.detach()

    JJT = torch.bmm(J, J.transpose(1, 2))
    JJT_reg = JJT + 1e-8 * torch.eye(K).expand_as(JJT)
    Jg = torch.bmm(J, grad_y.unsqueeze(-1))
    M_Jg = torch.linalg.solve(JJT_reg, Jg)
    normal = torch.bmm(J.transpose(1, 2), M_Jg).squeeze(-1)
    normal_frac = (normal.norm(dim=-1) / grad_y.norm(dim=-1).clamp(min=1e-12)).mean().item()

    return cos.item(), normal_frac, g_yhat.norm().item(), g_ytil.norm().item()


def train_pal(loss_at="yhat", epochs=EPOCHS, eval_seed=999, grad_every=50,
              switch_at=None, verbose=True):
    """Train PAL with f(yhat), f(ytilde), or switch midway.

    loss_at: "yhat" | "ytilde" | "switch"
    switch_at: epoch at which to flip to f(ytilde). Required if loss_at="switch".
    """
    assert loss_at in {"yhat", "ytilde", "switch"}
    if loss_at == "switch":
        assert switch_at is not None, "switch_at required for loss_at='switch'"

    torch.manual_seed(0)
    net = model()
    opt = torch.optim.Adam(net.parameters(), lr=1e-4)

    proj_delta, proj_lam_min = 1e-3, 1e-6
    feas_tol, disp_tol = 1e-7, 1e-1
    grad_clip = 20.0

    lambda_c = torch.zeros(2)
    lambda_d = torch.tensor(0.0)

    history = {k: [] for k in
        ["epoch", "f", "f_post1", "f_post10",
         "c_pre", "c_post", "c_post10", "lam_c", "lam_d"]}
    grad_diag = {k: [] for k in
        ["epoch", "cos_sim", "normal_frac", "g_yhat_norm", "g_ytilde_norm"]}

    for epoch in range(epochs):
        # decide what to use this epoch
        if loss_at == "switch":
            place = "yhat" if epoch < switch_at else "ytilde"
        else:
            place = loss_at

        x = sample_conditions(seed=epoch)
        y_hat = net(x)
        cfn = lambda yy: torch.stack(example_problem(x, yy)[3:5], dim=-1)

        _, c_pre, c_post_live, disp_live = lm_project_step(
            y_hat, cfn, delta=proj_delta, lam_min=proj_lam_min,
        )

        c_pre_res = c_pre.abs()
        c_post_res = c_post_live.abs()
        disp_mag = disp_live.pow(2).sum(dim=-1).add(1e-12).sqrt()

        if place == "yhat":
            f, *_ = example_problem(x, y_hat)
        else:
            y_tilde_live = y_hat - disp_live
            f, *_ = example_problem(x, y_tilde_live)

        constraint_loss   = (lambda_c.detach() * c_pre_res).sum(dim=-1).mean()
        displacement_loss = (lambda_d.detach() * disp_mag).mean()
        loss = f + constraint_loss + displacement_loss

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
        opt.step()

        with torch.no_grad():
            lambda_c = gated_ema_update3(lambda_c, c_post_res.mean(dim=0), feas_tol)
            lambda_d = gated_ema_update3(lambda_d, disp_mag.mean(),        disp_tol)

        # eval logging (every epoch)
        with torch.no_grad():
            x_e = sample_conditions(seed=eval_seed)
            y_e = net(x_e)
        cfn_e = lambda yy: torch.stack(example_problem(x_e, yy)[3:5], dim=-1)
        y_post1_e, c_pre_e, c_post_e, _ = lm_project_step(
            y_e.detach(), cfn_e, delta=proj_delta, lam_min=proj_lam_min,
        )
        y_iter = y_e.detach()
        for _ in range(10):
            y_iter, _, _, _ = lm_project_step(
                y_iter, cfn_e, delta=proj_delta, lam_min=proj_lam_min,
            )
        with torch.no_grad():
            c_post10_e = cfn_e(y_iter)
            f_e, *_      = example_problem(x_e, y_e)
            f_post1, *_  = example_problem(x_e, y_post1_e)
            f_post10, *_ = example_problem(x_e, y_iter)
            history["epoch"   ].append(epoch)
            history["f"       ].append(f_e.item())
            history["f_post1" ].append(f_post1.mean().item())
            history["f_post10"].append(f_post10.mean().item())
            history["c_pre"   ].append(c_pre_e.abs().mean(dim=0).tolist())
            history["c_post"  ].append(c_post_e.abs().mean(dim=0).tolist())
            history["c_post10"].append(c_post10_e.abs().mean(dim=0).tolist())
            history["lam_c"   ].append(lambda_c.tolist())
            history["lam_d"   ].append(lambda_d.item())

        # gradient diagnostics
        if epoch % grad_every == 0 or epoch == epochs - 1:
            cos, nfrac, ng_y, ng_t = grad_diagnostics(
                net, x_e, cfn_e, proj_delta, proj_lam_min,
            )
            grad_diag["epoch"        ].append(epoch)
            grad_diag["cos_sim"      ].append(cos)
            grad_diag["normal_frac"  ].append(nfrac)
            grad_diag["g_yhat_norm"  ].append(ng_y)
            grad_diag["g_ytilde_norm"].append(ng_t)

        if verbose and (epoch % 1000 == 0 or epoch == epochs - 1):
            i = -1
            tag = f"{loss_at}@{place}"
            print(f"[{tag:>14s}] ep {history['epoch'][i]:4d}  f={history['f'][i]:.4f}  "
                  f"|c_post1|={[f'{v:.4f}' for v in history['c_post'][i]]}  "
                  f"lam_c={[f'{v:.3f}' for v in history['lam_c'][i]]}  "
                  f"lam_d={history['lam_d'][i]:.3f}")

    return history, grad_diag


In [ ]:
SWITCH_AT = EPOCHS // 2

print("=" * 70)
print("PAL with f(y_hat)  -- standard placement")
print("=" * 70)
H_yhat,   G_yhat   = train_pal(loss_at="yhat")

print()
print("=" * 70)
print("PAL with f(y_tilde^(1))  -- broken placement (routes through projector)")
print("=" * 70)
H_ytilde, G_ytilde = train_pal(loss_at="ytilde")

print()
print("=" * 70)
print(f"PAL switching y_hat -> y_tilde at epoch {SWITCH_AT}")
print("=" * 70)
H_switch, G_switch = train_pal(loss_at="switch", switch_at=SWITCH_AT)


Comparison figure for the three variants, saved to figures/fhat_vs_ftilde.pdf.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from collections import deque

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
})

C_H1 = "#8B3EBC"   # purple, h_1
C_H2 = "#4FC3F7"   # cyan, h_2
C_F1 = "#A33B3F"   # dark warm red, f(y_hat) / ||grad_theta f(y_hat)||
C_F2 = "#F4A461"   # warm light orange, f(y_tilde) / ||grad_theta f(y_tilde^(1))||


def to_array(H):
    return np.column_stack([
        H["epoch"], H["f"], H["f_post1"], H["f_post10"],
        np.array(H["c_pre"]), np.array(H["c_post"]),
        np.array(H["c_post10"]), np.array(H["lam_c"]),
        np.array(H["lam_d"]),
    ])

A_yhat   = to_array(H_yhat)
A_ytilde = to_array(H_ytilde)
A_switch = to_array(H_switch)

methods = [
    (r"PAL: $f(\hat{y})$",                                                  A_yhat,   G_yhat,   None,      "yhat_active"),
    (r"PAL: $f(\tilde{y}^{(1)})$",                                          A_ytilde, G_ytilde, None,      "ytilde_active"),
    (fr"PAL: $f(\hat{{y}}) \to f(\tilde{{y}}^{{(1)}})$ at $t={SWITCH_AT}$", A_switch, G_switch, SWITCH_AT, "switch"),
]


def _moving_avg(vals, window):
    if window <= 1 or len(vals) <= window:
        return list(vals)
    out, buf = [], deque(maxlen=window)
    for v in vals:
        buf.append(v)
        out.append(sum(buf) / len(buf))
    return out


fig, axes = plt.subplots(4, 3, figsize=(6.5, 5.0),
                         sharex="col", sharey="row")
fig.subplots_adjust(hspace=0.30, wspace=0.18,
                    left=0.12, right=0.985, top=0.95, bottom=0.07)

row_labels = [
    r"$h(\hat{y})$",
    r"$h(\tilde{y}^{(1)})$",
    r"$f$",
    r"$\Vert \nabla_\theta f \Vert$",
]

# shared y range for residual rows
res_vals = []
for _, A, _, _, _ in methods:
    for c in (4, 5, 6, 7):
        res_vals.extend(max(v, 1e-10) for v in A[:, c])
res_lo, res_hi = min(res_vals) * 0.5, max(res_vals) * 2.0

# shared y range for grad-norm row
g_vals = []
for _, _, G, _, _ in methods:
    g_vals.extend(max(v, 1e-12) for v in G["g_yhat_norm"])
    g_vals.extend(max(v, 1e-12) for v in G["g_ytilde_norm"])
g_lo, g_hi = min(g_vals) * 0.3, max(g_vals) * 3.0

# shared y range for objective row (log), set explicitly so leftmost matches
f_vals = []
for _, A, _, _, _ in methods:
    f_vals.extend([v for v in A[:, 1] if v > 0])
    f_vals.extend([v for v in A[:, 3] if v > 0])
f_lo, f_hi = min(f_vals) * 0.85, max(f_vals) * 0.70


for col, (name, A, G, sw_epoch, mode) in enumerate(methods):
    epochs = np.array(A[:, 0])
    w = max(1, len(epochs) // 40)

    # Row 0: pre h
    ax = axes[0, col]
    h1 = np.maximum(A[:, 4], 1e-10)
    h2 = np.maximum(A[:, 5], 1e-10)
    ax.plot(epochs, h1, color=C_H1, alpha=0.35, linewidth=0.5)
    ax.plot(epochs, h2, color=C_H2, alpha=0.35, linewidth=0.5)
    ax.plot(epochs, _moving_avg(list(h1), w), color=C_H1, linewidth=1.1)
    ax.plot(epochs, _moving_avg(list(h2), w), color=C_H2, linewidth=1.1)
    ax.set_yscale("log"); ax.set_ylim(res_lo, res_hi)
    ax.set_title(name)

    # Row 1: post-1 h
    ax = axes[1, col]
    h1 = np.maximum(A[:, 6], 1e-10)
    h2 = np.maximum(A[:, 7], 1e-10)
    ax.plot(epochs, h1, color=C_H1, alpha=0.35, linewidth=0.5)
    ax.plot(epochs, h2, color=C_H2, alpha=0.35, linewidth=0.5)
    ax.plot(epochs, _moving_avg(list(h1), w), color=C_H1, linewidth=1.1, label=r"$h_1$")
    ax.plot(epochs, _moving_avg(list(h2), w), color=C_H2, linewidth=1.1, label=r"$h_2$")
    ax.set_yscale("log"); ax.set_ylim(res_lo, res_hi)

    # Row 2: objective f(yhat) (red) and f(ytilde^(10)) (orange)
    ax = axes[2, col]
    f_yhat   = np.array(A[:, 1])
    f_yt10   = np.array(A[:, 3])
    f_yhat_ma = np.array(_moving_avg(list(f_yhat), w))
    f_yt10_ma = np.array(_moving_avg(list(f_yt10), w))

    if mode == "yhat_active":
        ax.plot(epochs, f_yhat, color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(epochs, f_yt10, color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(epochs, f_yhat_ma, color=C_F1, lw=1.1, label=r"$f(\hat{y})$")
        ax.plot(epochs, f_yt10_ma, color=C_F2, lw=0.75, ls="--", alpha=0.65, label=r"$f(\tilde{y}^{(10)})$")
    elif mode == "ytilde_active":
        ax.plot(epochs, f_yhat, color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(epochs, f_yt10, color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(epochs, f_yhat_ma, color=C_F1, lw=0.75, ls="--", alpha=0.65)
        ax.plot(epochs, f_yt10_ma, color=C_F2, lw=1.1)
    else:  # switch
        pre = epochs < sw_epoch
        post = ~pre
        ax.plot(epochs[pre],  f_yhat[pre],     color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(epochs[post], f_yt10[post],    color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(epochs[pre],  f_yhat_ma[pre],  color=C_F1, lw=1.1)
        ax.plot(epochs[post], f_yt10_ma[post], color=C_F2, lw=1.1)
    ax.set_yscale("log"); ax.set_ylim(f_lo, f_hi)
    ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0, subs=(1.0,), numticks=15))
    ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=tuple(i/10 for i in range(2,10)), numticks=50))
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())

    # Row 3: theta-gradient norms
    ax = axes[3, col]
    g_eps = np.array(G["epoch"])
    gy = np.maximum(G["g_yhat_norm"], 1e-12)
    gt = np.maximum(G["g_ytilde_norm"], 1e-12)
    gw = max(1, len(g_eps) // 40)
    gy_ma = np.array(_moving_avg(list(gy), gw))
    gt_ma = np.array(_moving_avg(list(gt), gw))

    if mode == "yhat_active":
        ax.plot(g_eps, gy, color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(g_eps, gt, color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(g_eps, gy_ma, color=C_F1, lw=1.1, label=r"$\Vert\nabla_\theta f(\hat{y})\Vert$")
        ax.plot(g_eps, gt_ma, color=C_F2, lw=0.75, ls="--", alpha=0.65, label=r"$\Vert\nabla_\theta f(\tilde{y}^{(1)})\Vert$")
    elif mode == "ytilde_active":
        ax.plot(g_eps, gy, color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(g_eps, gt, color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(g_eps, gy_ma, color=C_F1, lw=0.75, ls="--", alpha=0.65)
        ax.plot(g_eps, gt_ma, color=C_F2, lw=1.1)
    else:  # switch
        pre = g_eps < sw_epoch
        post = ~pre
        ax.plot(g_eps[pre],  gy[pre],     color=C_F1, alpha=0.35, lw=0.5)
        ax.plot(g_eps[post], gt[post],    color=C_F2, alpha=0.35, lw=0.5)
        ax.plot(g_eps[pre],  gy_ma[pre],  color=C_F1, lw=1.1)
        ax.plot(g_eps[post], gt_ma[post], color=C_F2, lw=1.1)
    ax.set_yscale("log"); ax.set_ylim(g_lo, g_hi)
    ax.set_xlabel("Epoch")

    # mark switch epoch on every panel of the switch column
    if sw_epoch is not None:
        for r in range(4):
            axes[r, col].axvline(sw_epoch, color="k", lw=0.8, ls=":", alpha=0.6)


for row, lbl in enumerate(row_labels):
    axes[row, 0].set_ylabel(lbl, fontsize=10)
    for col in range(3):
        axes[row, col].grid(True, which="both", alpha=0.25)

# Align y-labels across rows so they don't shift with tick-label width
fig.align_ylabels(axes[:, 0])

# Legends: row-0 in rightmost (residuals h_1/h_2),
#          row-2 and row-3 in leftmost (red/orange pair shown solid+dashed there)
axes[1, 0].legend(loc="upper right")
axes[2,  0].legend(loc="upper right")
axes[3,  0].legend(loc="center right")

plt.savefig("figures/fhat_vs_ftilde.pdf", bbox_inches="tight")
plt.show()
